# SBND RWM True Bucket: Step 1 Trace Fits

- Fits every trace of the extraction (`inputs/traces_run{1,2}.csv` rows with `fit_row` ≥ 0, in that order): 81 buckets, positions, period, σ, QC
- Writes `OUT_DIR/run{1,2}/trace_fits.csv` (one row per trace) + `trace_buckets.npz` (per-bucket results, same rows)

## 1. Setup

- Set the paths in the Setup cell: `STORE` = stored raw RWM traces, `OUT_DIR` = output folder

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import numpy as np
import pandas as pd

import sys
sys.path.insert(0, '../lib')
import paths
import ad_trace as at

# ===== Set these paths =====
STORE = {1: '/path/to/run1_rwm_traces', 2: '/path/to/run2_rwm_traces'}   # stored raw RWM traces: folders of run_<DAQ run>.npz
OUT_DIR = '../output'   # output folder, the same in every notebook
paths.set_paths(store=STORE, out=OUT_DIR)

## 2. Fit

- `qc`: 'good', the failed guard, or 'stuck readout' (trace bytes identical to another trace)

In [ ]:
for run in (1, 2):
    lst = pd.read_csv(f'{paths.INPUTS}/traces_run{run}.csv')
    sel = lst[lst['fit_row'] >= 0].sort_values('fit_row')
    d = at.ensure(paths.STORE[run], sel[['run', 'event', 'ts_ns']], allow_fetch=False)
    assert len(d['run']) == len(sel) and (d['status'] == 1).all(), f'Run {run}: traces missing from the store'
    rows, pb = at.fit_entries(d, 'sym')
    fits = at.add_qc(pd.DataFrame(rows))
    paths.write_csv(fits, paths.fits_csv(run))
    np.savez_compressed(paths.buckets_npz(run), **{k: np.array(v) for k, v in pb.items()})
    print(f'Run {run}: {len(fits)} traces; ' + ', '.join(f'{k}: {v}' for k, v in fits['qc'].value_counts().items()))